In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import numpy as np
import matplotlib.pyplot as plt

# Load the data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mrtontrnok/5-vehichles-for-multicategory-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the '5-vehichles-for-multicategory-classification' dataset.
Path to dataset files: /kaggle/input/5-vehichles-for-multicategory-classification


In [ ]:
path = "/kaggle/input/5-vehichles-for-multicategory-classification/dataset"

In [ ]:
TRAIN_DIR = f"{path}/train"
TEST_DIR = f"{path}/test"
VAL_DIR = f"{path}/validation"

print(f"Train directory: {TRAIN_DIR}")
print(f"Test directory: {TEST_DIR}")
print(f"Validation directory: {VAL_DIR}")

Train directory: /kaggle/input/5-vehichles-for-multicategory-classification/dataset/train
Test directory: /kaggle/input/5-vehichles-for-multicategory-classification/dataset/test
Validation directory: /kaggle/input/5-vehichles-for-multicategory-classification/dataset/validation


# Data Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation and normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

# Prepare data generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(192, 192),
    batch_size=32,
    class_mode='categorical')

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(192, 192),
    batch_size=32,
    class_mode='categorical')

validation_generator = validation_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(192, 192),
    batch_size=32,
    class_mode='categorical')

print("Data generators prepared.")

Found 5418 images belonging to 5 classes.
Found 708 images belonging to 5 classes.
Found 709 images belonging to 5 classes.
Data generators prepared.


# CNN Model

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(192, 192, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_generator.num_classes, activation='softmax') # Use the number of classes from the generator
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy', # Use categorical_crossentropy for one-hot encoded labels
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 190, 190, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 95, 95, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 93, 93, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 46, 46, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 44, 44, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 22, 22, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 61952)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     7,929,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,023,877 (30.61 MB)

 Trainable params: 8,023,877 (30.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=30, # You can adjust the number of epochs
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
169/169 ━━━━━━━━━━━━━━━━━━━━ 450s 3s/step - accuracy: 0.3471 - loss: 1.5022 - val_accuracy: 0.5057 - val_loss: 1.4173
Epoch 2/30
  1/169 ━━━━━━━━━━━━━━━━━━━━ 5:36 2s/step - accuracy: 0.4375 - loss: 1.6854

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


169/169 ━━━━━━━━━━━━━━━━━━━━ 21s 110ms/step - accuracy: 0.4375 - loss: 1.6854 - val_accuracy: 0.5909 - val_loss: 1.0645
Epoch 3/30
169/169 ━━━━━━━━━━━━━━━━━━━━ 476s 3s/step - accuracy: 0.5855 - loss: 1.0854 - val_accuracy: 0.6435 - val_loss: 0.9513
Epoch 4/30
169/169 ━━━━━━━━━━━━━━━━━━━━ 21s 104ms/step - accuracy: 0.5938 - loss: 1.0981 - val_accuracy: 0.6435 - val_loss: 0.9459
Epoch 5/30
169/169 ━━━━━━━━━━━━━━━━━━━━ 502s 3s/step - accuracy: 0.6580 - loss: 0.8900 - val_accuracy: 0.6790 - val_loss: 0.8686
Epoch 6/30
169/169 ━━━━━━━━━━━━━━━━━━━━ 21s 111ms/step - accuracy: 0.6562 - loss: 0.9699 - val_accuracy: 0.6832 - val_loss: 0.8956
Epoch 7/30
169/169 ━━━━━━━━━━━━━━━━━━━━ 441s 3s/step - accuracy: 0.6939 - loss: 0.8216 - val_accuracy: 0.7045 - val_loss: 0.7884
Epoch 8/30
169/169 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.6562 - loss: 0.7780 - val_accuracy: 0.7074 - val_loss: 0.7804
Epoch 9/30
169/169 ━━━━━━━━━━━━━━━━━━━━ 442s 3s/step - accuracy: 0.7333 - loss: 0.7130 - val_accuracy

In [ ]:
model.evaluate(test_generator)

23/23 ━━━━━━━━━━━━━━━━━━━━ 16s 675ms/step - accuracy: 0.7341 - loss: 0.9857


[0.8771616816520691, 0.7429378628730774]

In [ ]:
model.save("my_cnn_model.keras")
print("Model saved successfully!")

Model saved successfully!


In [ ]:
from tensorflow.keras.models import load_model
import os
import numpy as np
from tensorflow.keras.preprocessing import image

# Load the saved model
loaded_model = load_model("my_cnn_model.keras")

# Define the image path(s) you want to predict on
# You can provide a single path as a string, or multiple paths as a list of strings
image_paths_to_predict = ["/kaggle/input/5-vehichles-for-multicategory-classification/dataset/validation/motorcycle/50.png"] # Replace with your absolute path(s)

# Process each image path
for image_path_to_predict in image_paths_to_predict:
    print(f"Predicting on image: {image_path_to_predict}")

    # Load and preprocess the image
    img = image.load_img(image_path_to_predict, target_size=(192, 192))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Rescale the image

    # Make a prediction
    predictions = loaded_model.predict(img_array)

    # Get the class labels from the validation generator (assuming it's still available)
    # If not, you might need to define class_labels manually based on your dataset
    # class_labels = ['bus', 'car', 'motorcycle', 'train', 'truck']
    if 'validation_generator' in globals():
        class_labels = list(validation_generator.class_indices.keys())
    else:
        # Fallback if validation_generator is not available
        print("Warning: validation_generator not found. Using hardcoded class labels.")
        class_labels = ['bus', 'car', 'motorcycle', 'train', 'truck']


    # Display the results
    print("\nPrediction probabilities:")
    for i, probability in enumerate(predictions[0]):
        print(f"{class_labels[i]}: {probability:.4f} ({probability*100:.2f}%)")

    predicted_class_index = np.argmax(predictions)
    predicted_class_label = class_labels[predicted_class_index]
    print(f"\nPredicted class: {predicted_class_label}")
    print("-" * 30) # Separator for multiple predictions

Predicting on image: /kaggle/input/5-vehichles-for-multicategory-classification/dataset/validation/motorcycle/50.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step

Prediction probabilities:
bus: 0.0000 (0.00%)
car: 0.0037 (0.37%)
motorcycle: 0.9963 (99.63%)
train: 0.0000 (0.00%)
truck: 0.0000 (0.00%)

Predicted class: motorcycle
------------------------------
